# FloodA5 notebook 4 — the sub-grid (SGS) model on a strip with known answers

On the Carlisle flood the sub-grid (SGS) model stores far more water than the
other models on the same A5 mesh: 26.7 Mm³ at the Eden peak, against 10.5
for the full shallow-water reference (SWE) and 8.0 for local-inertial, with
its water surface a median 3 m higher (`notebooks/03_carlisle.ipynb`). On the
planar embankment it is 1–1.5 m deep at the inflow where the standard model
is 0.3 m deep (`notebooks/01_planar.ipynb`). Two explanations were possible:

- **measurement** — SGS keeps the water that lies below a cell's mean bed
  (in channels and hollows), which a grid of cell-mean elevations smooths
  away, so its "storage" counts more of the same water; or
- **conveyance** — SGS lets water out of a cell more slowly than it should,
  so the water backs up.

This notebook answers that on a synthetic case built so that every answer is
known in advance: a long, narrow strip on a uniform slope with a straight
channel narrower than a mesh cell (case `sgs_strip`). It shows

1. the geometry, the meshes and the runs, with their exact commands (§1–2);
2. the analytic answers — normal depth, storage per metre, arrival times —
   and what they say about measuring storage (§3);
3. how SGS represents the sub-cell channel, by comparing its tables with the
   terrain directly, without a simulation (§4);
4. the runs against the analytic answers and a 2.5 m reference that resolves
   the channel (§5);
5. why SGS holds water back, with the code responsible (§6), and
   conclusions (§7).

**In short:** storage is measured correctly — the water SGS holds is real
water, held back by too little conveyance. Overbank, an SGS face's flow area
stops growing once the water rises above the cell's highest DEM sample; in
bank, SGS's default limiter caps the discharge of a cell by its volume.

**How to run it.** Open it with the kernel **"Julia FloodA5 1.12"** (set-up:
`notebooks/README.md`) and run all cells. The first run makes the case: the
DEM, five meshes and 16 simulations, about 40 minutes with two runs at a time
(the 2.5 m SWE reference is the longest) in `test/case_runs/sgs_strip/` (not
tracked by git). Later runs reuse them and take a minute or two.

**Words used throughout.** *δ*: the water level above the local floodplain,
η − z_f(x) (negative: in bank). *Normal (uniform) flow*: the steady state on
a long uniform slope, the water surface parallel to the bed. *LI*:
local-inertial, `--flow-model standard`. *Cell-mean bed*: the mean DEM
elevation of a cell, which is all a standard mesh knows of its terrain.

## 0. Set-up

The cell below finds the repository from the Julia project the kernel was
started with and loads the three modules the notebooks use: `FA`
(`analysis/FloodA5Analysis.jl`, reading outputs), `FC` (`cases/FloodA5Cases.jl`,
the test-case registry; the case is defined in `cases/registry/sgs_strip.jl`)
and `FP` (`visualisation/FloodA5Plots.jl`, the figure style).

`RUN_CASES = false` only analyses runs that already exist. `PARALLEL` runs
that many simulations at once, each with its own thread count (2 for the
20 m and A5 meshes, 4 for the 2.5 m reference: at most 8 threads with
`PARALLEL = 2`).

In [ ]:
# The kernel starts Julia with --project=<repository> (or <repository>/ci):
# walk up from that project to the directory holding FloodModel.jl.
const REPO = let d = dirname(something(Base.active_project(), joinpath(pwd(), "Project.toml")))
    while !isfile(joinpath(d, "FloodModel.jl"))
        dirname(d) == d && error("no FloodA5 repository above $(Base.active_project())")
        d = dirname(d)
    end
    d
end
ENV["FLOODA5_NO_VIS"] = "1"            # the runs need no viewer

include(joinpath(REPO, "analysis", "FloodA5Analysis.jl"));     const FA = FloodA5Analysis
include(joinpath(REPO, "cases", "FloodA5Cases.jl"));           const FC = FloodA5Cases
include(joinpath(REPO, "visualisation", "FloodA5Plots.jl"));   const FP = FloodA5Plots
using CairoMakie, Printf, Statistics
FP.activate!()                         # the FloodA5 figure style

const RUN_CASES = true                 # false: only analyse existing runs
const PARALLEL  = 2                    # simulations at once (each with its own thread count)

println("repository : ", REPO)
println("version    : ", strip(read(joinpath(REPO, "VERSION"), String)))
println("case runs  : ", FC.case_root())

Two small helpers: `mdtable` shows a table (as HTML), `f` formats a number.

In [ ]:
"An HTML table from a header and rows of already formatted values. (A Markdown table also carries
a LaTeX copy, a `tabular` environment, which VS Code's KaTeX renderer cannot draw.)"
function mdtable(header, rows)
    esc(x) = replace(string(x), "&" => "&amp;", "<" => "&lt;", ">" => "&gt;")
    cell(tag, x) = "<$tag style=\"text-align:right\">" * esc(x) * "</$tag>"
    s = "<table><thead><tr>" * join(cell("th", h) for h in header) * "</tr></thead><tbody>"
    for r in rows
        s *= "<tr>" * join(cell("td", v) for v in r) * "</tr>"
    end
    return HTML(s * "</tbody></table>")
end
"`x` to `d` decimals, or as 1.2e-05 if it would round to zero."
f(x, d = 3) = !isfinite(x) ? string(x) :
              (x != 0 && abs(x) < 0.5 * 10.0^-d) ? @sprintf("%.1e", x) : string(round(x; digits = d))
nothing

## 1. The case: a sub-cell channel in a long sloping flume

### Geometry

The terrain (`cases/scripts/sgs_strip_dem.jl`, a 0.5 m DEM) is chosen so that
every answer can be written down without a model:

- a **flume** 4000 m long (x, west → east) and 60 m wide (y), its floodplain a
  plane sloping down the flume at **S = 0.001** (1 m per km);
- a **rectangular channel 5 m wide and 1 m deep** cut along the centre line;
- **no walls**: the floodplain plane continues beyond the strip, and the
  mesh's own edges are closed boundaries except at the downstream end;
- an **outlet reach**: below x = 3800 m the bed steepens to 2.1%. The flow
  passes through critical depth at the break, so whatever each model does at
  its outflow boundary cannot travel back up the flume;
- Manning's **n = 0.03** everywhere;
- a constant inflow (after a 10-minute ramp) into the channel 31 m from the
  upstream end: **3 m³/s**, which stays in the channel (in bank), or
  **20 m³/s**, which spills onto the floodplain (overbank).

The DEM stores the bed at its nodes and the model samples it by bilinear
interpolation, so the model sees exactly this surface except that each
channel wall becomes a ramp 0.5 m wide centred on it. The analytic answers
below use that same surface (`cases/scripts/sgs_strip_geometry.jl`).

### Meshes and runs

| Mesh | Cells | What it sees of the channel |
|---|---|---|
| Cartesian 20 m (`cart20`) | 200 × 3 rows | the channel is a quarter of the middle row; SGS mesh with sub-grid tables, standard mesh with cell-mean beds |
| A5 res 18 (`a5r18`) | pentagons of ~22 m covering the 60 m strip, so its edges are ragged (every cell the strip outline touches: a mean width of 95 m) | as above, but the channel crosses cells at every angle and offset |
| Cartesian 2.5 m (`cart2p5`) | 1600 × 24 | the channel is two cells wide: the **fine reference**, cell-mean beds |

On each mesh the case runs the models at both discharges: **SGS** (the
default limiter, and `--limiter positivity`, a diagnostic explained in §6),
**local-inertial** (`--flow-model standard`) and **SWE** on 20 m; SGS and
local-inertial on A5; local-inertial and SWE on 2.5 m. Coarse runs last 8 h
(a 20 m cell-mean grid fills slowly at 3 m³/s: the water spreads as a thin
sheet over its 60 m width), the 2.5 m runs 4 h; all write a frame every 5
minutes.

### What is measured

Over the **measuring reach x = 1000–2000 m**, at the last frame:

- **δ**, the water level above the local floodplain, η − z_f(x), in the
  channel cells (and separately in the floodplain cells, if wet). At uniform
  flow it is the same everywhere; in bank δ < 0 (−0.27 m means 0.73 m of
  water in the 1 m channel);
- **storage per metre** of flume (m²): Σ volume / 1000 m, the water a model
  holds; and the **volume above the cell-mean bed**, Σ area·max(η − z̄, 0), what
  a grid of cell-mean elevations would hold at the same water level — the
  two ways of measuring storage the user asked about;
- **steadiness**: the change of the stored volume over the last hour, as a
  fraction of the inflow;
- **arrival**: the time at which the ±50 m window around x = 1000 m and
  2000 m first holds half the 2-D normal storage.

In [ ]:
ss = FC.case("sgs_strip")
FC.show_case(ss)

Reading the commands:

- **prep dem** writes the DEM, the parameters file, the AOI (the flume
  outline, shrunk by 1 cm so the Cartesian generator makes exactly 200 × 3
  cells) and the boundary file `sgs_strip_bc.geojson` (`--bc-file`): a
  *Closed* line round the flume, then a *ZeroGradient* (open) line across the
  downstream end. A feature applies to boundary cells within 1.5 cell
  diameters of it, the later one winning, so the last 1.5 cell diameters of the strip are
  open on every side (the last two columns of the 20 m grid).
- **prep cart20_sgs / a5r18_sgs** build meshes *with SGS tables*
  (`--flow-model sgs --mesh-only`); **cart20_standard / a5r18_standard /
  cart2p5_standard** meshes with cell-mean beds only. `--cartesian-lat-ref 51`
  makes the Cartesian cells square in the DEM's own metric, so rows run
  exactly along the channel.
- **prep tables_cart20 / tables_a5r18** compare each SGS mesh's tables with
  the terrain, without a simulation (`cases/scripts/sgs_strip_tables.jl`;
  §4).
- **run …** — `--flow-model` is always explicit; `--inflow-point lat,lon,file`
  (latitude first) feeds the hydrograph to the cell containing the point;
  `--bc-file` sets the boundaries; `--dt-max 10` caps the time step (the
  Courant condition sets it in practice).

### Run and check

`FC.run_case` makes whatever is missing (first time: the DEM, five meshes,
the table diagnostics and 16 simulations — about 5 minutes of preparation,
a minute per coarse run and 5–25 minutes per 2.5 m run; ~35 minutes with
`PARALLEL = 2`), then computes the metrics and runs the checks. The checks
state what a **correct** model must do. Several SGS checks fail at present;
they are kept as they are, as expected failures, until SGS is fixed (§6).

In [ ]:
t0 = time()
ok_ss = RUN_CASES ? FC.run_case(ss; parallel = PARALLEL) : FC.check(ss)
@printf("\nsgs_strip: %s (%.0f s)\n", ok_ss ? "all checks pass" : "SOME CHECKS FAIL", time() - t0)
m = FC.analyse(ss; io = devnull)        # the metrics (also in test/case_runs/sgs_strip/metrics.toml)
p = FC.sgs_strip_params(FC.case_dir(ss))
a = m["analytic"]
nothing

## 2. The terrain and the meshes

The cross-section is the same at every x above the outlet reach (relative to
the floodplain level z_f(x) = 10 − 0.001·x m). The figure shows it with the
2-D normal levels of §3 and, below, the three meshes over a 200 m stretch of
the measuring reach with the channel shaded.

In [ ]:
const LABEL = Dict("cart20_sgs" => "SGS, Cartesian 20 m", "cart20_sgspos" => "SGS (positivity), Cartesian 20 m",
                   "cart20_std" => "LI, Cartesian 20 m", "cart20_swe" => "SWE, Cartesian 20 m",
                   "a5r18_sgs" => "SGS, A5 res 18", "a5r18_std" => "LI, A5 res 18",
                   "fine_std" => "LI, Cartesian 2.5 m", "fine_swe" => "SWE, Cartesian 2.5 m")
const QS = (3.0, 20.0); qtag(Q) = "q" * string(round(Int, Q))
label(job) = LABEL[rsplit(job, '_'; limit = 2)[1]] * ", " * replace(rsplit(job, '_'; limit = 2)[2], "q" => "") * " m³/s"
runs = Dict(j.name => FA.load_run(FC.job_output(ss, j); label = label(j.name)) for j in ss.jobs)
xy(r) = FA.local_xy(r.mesh, 0.0, 51.0)          # the strip's frame: x along the flume, y across

fig = Figure(size = (900, 820))
ax = Axis(fig[1, 1]; xlabel = "y (m across the flume)", ylabel = "bed − z_f (m)",
          title = "Cross-section of the strip (any x < 3800 m) with the 2-D normal levels")
ys = range(-32, 32; length = 2001)
band!(ax, ys, fill(-1.2, length(ys)), [FC.sgs_strip_zrel(p, y) for y in ys]; color = (FP.INK2, 0.25))
lines!(ax, ys, [FC.sgs_strip_zrel(p, y) for y in ys]; color = FP.INK, linewidth = 1.5, label = "bed (DEM as sampled)")
for (k, Q) in enumerate(QS)
    δ = a["delta_2d_$(qtag(Q))"]
    lines!(ax, [-30, 30], [δ, δ]; color = FP.SERIES[k], linewidth = 2, label = "2-D normal level, $(Int(Q)) m³/s")
end
vlines!(ax, [-10, 10]; color = FP.INK2, linestyle = :dash, linewidth = 0.8)
text!(ax, 0, 1.2; text = "dashed: the 20 m Cartesian rows", align = (:center, :bottom), fontsize = 11, color = FP.INK2)
ylims!(ax, -1.2, 1.6); xlims!(ax, -32, 32); axislegend(ax; position = :lb, framevisible = false, backgroundcolor = (:white, 0.8))

for (k, (job, title)) in enumerate((("cart20_std_q3", "Cartesian 20 m"), ("a5r18_std_q3", "A5 res 18"),
                                     ("fine_std_q3", "Cartesian 2.5 m")))
    r = runs[job]
    axm = Axis(fig[k + 1, 1]; aspect = DataAspect(), title = "$title mesh over x = 1400–1600 m (channel shaded)",
               xlabel = k == 3 ? "x (m along the flume)" : "", ylabel = "y (m)")
    poly!(axm, Rect(1400, -2.5, 200, 5); color = (FP.SERIES[1], 0.35))
    for P in r.mesh.polygons
        x, y = FA.local_xy(P[1, :], P[2, :], 0.0, 51.0)
        (maximum(x) < 1400 || minimum(x) > 1600) && continue
        lines!(axm, x, y; color = FP.INK, linewidth = 0.6)
    end
    xlims!(axm, 1400, 1600); ylims!(axm, -55, 55)
end
fig

## 3. The analytic answers

At **uniform flow** the water surface is parallel to the bed and friction
balances gravity, Q = K(δ)·√S / n, where the conveyance K depends on how the
section is represented. Three representations matter here (all in
`cases/scripts/sgs_strip_geometry.jl`, solved for δ by bisection):

- **2-D** — the depth-averaged equations with no lateral exchange: every strip
  dy of the section carries its own Manning flow, K = ∫ h(y)^(5/3) dy. This is
  what a fine 2-D model converges to (lateral momentum exchange vanishes at
  uniform flow without turbulence closure), so it is the reference for
  everything below.
- **cell-mean (`cells20`)** — the three 20 m rows each with its mean bed, as a
  standard 20 m grid sees them: K = Σ 20·(δ − z̄)^(5/3). The middle row's mean
  bed lies 0.25 m below the floodplain (the channel's 5 m² spread over 20 m).
- **lumped (`lumped20`)** — each 20 m row with its *true* section but one
  hydraulic radius, K = Σ A^(5/3)/P_w^(2/3) (A the flow area, P_w the wetted
  width). This is the SGS R–A edge formula, `_manning_flux_ra_corrected`
  (`surfacewater/flow2d.jl`), evaluated with *exact* edge tables: what SGS
  should give on the 20 m strip if its tables were right.

**Storage per metre** is the water volume per metre of flume at that level;
for the cell-mean grid it is the volume above the cell-mean beds.
**Arrival**: on a dry bed a kinematic wave front (a shock) moves at Q/A_n, A_n
the normal-depth storage per metre, so a station x is reached at
t = 300 s + A_n·(x − 31 m)/Q (the 10-minute ramp delays the volume by 300 s).

In [ ]:
rows = []
for Q in QS, (lab, key) in (("2-D (fine-grid limit)", "2d"), ("cell-mean 20 m rows (standard)", "cells20"),
                            ("lumped 20 m rows (SGS, exact tables)", "lumped20"), ("2-D, the A5 strip's mean width", "a5w"))
    q = qtag(Q)
    push!(rows, (Int(Q), lab, f(a["delta_$(key)_$q"]), f(a["delta_$(key)_$q"] + p.D), f(a["store_$(key)_$q"], 2),
                 f(a["store_$(key)_$q"] / a["store_2d_$q"], 3),
                 round(Int, a["t_arrive_$(key)_$(q)_x1000"]), round(Int, a["t_arrive_$(key)_$(q)_x2000"])))
end
mdtable(["Q (m³/s)", "section", "δ (m)", "channel depth (m)", "storage (m²/m)", "÷ 2-D", "arrival x=1000 (s)", "x=2000 (s)"], rows)

### The storage-measurement question

A grid of cell-mean elevations removes the small topographic variations — but
it does not remove their *volume* once they are under water. The cell-mean
bed is the mean of the terrain, so for a water level above the highest point
of a cell, the water above the mean bed, area·(η − z̄), is exactly the water
volume ∫(η − z) dA. Here, above the floodplain (δ ≥ 0), the volume above the
20 m cell-mean beds equals the water volume (the table's 29.52 m²/m at
δ = 0.409 m is both), and the 20 m standard grid stores only 8% more than the
2-D answer at 20 m³/s — because its conveyance is lower (§3 table, +0.04 m),
not because of how it measures storage.

The two measures differ only where a cell is partly dry. In bank (3 m³/s) the
true water volume is 3.57 m²/m; a 20 m cell-mean grid has no channel, spreads
the flow over its whole 20 m row and then onto the side rows, and stores
7.71 m²/m (2.2×). So *at low flow* SGS should store **less** than a standard
grid, and at high flow about the same. A model that stores 2.5 times as much
as SWE at a flood peak, with water 3 m higher, is not explained by the
storage measure: it is holding water back. §5 shows that SGS does exactly
that here, and §4 and §6 say why.

## 4. How SGS represents the sub-cell channel

An SGS mesh carries, for every cell, tables built from the DEM
(`build_sgs_tables!`, `mesh/A5Grid.jl`):

- a **hypsometric curve**: the stored volume V(η) and wetted area at 100 water
  levels η spaced evenly from the cell's lowest to its highest DEM sample
  (z_min to z_max, from 512 Halton points); the model inverts it to get η
  from the volume (`wse_from_volume`) and, above z_max, extends it with the
  full cell area;
- for every **face** shared with a neighbour, the **flow area A(η)** and the
  **wetted width P(η)** of the face's cross-section, sampled along the face
  and tabulated at *the same 100 levels* of the cell's own curve
  (`sgs_edge_area_curve`, `sgs_edge_perim_curve`).

The flux kernel (`_face_flux_sgs!`, `surfacewater/kernels.jl`) takes the
higher of the two cells' levels, looks up A and R = A/P in *both* cells'
tables (`edge_area_radius`) and uses the averages, A = ½(A_i + A_j) and
R = ½(R_i + R_j), in Manning's law. `edge_area_radius` returns the value at
the top knot for any level above it: **the flow area of a face stops growing
when the water rises above the cell's highest DEM sample.** For a
floodplain cell on a gentle slope that is a few centimetres above its lowest
point (on this strip, a 20 m cell spans 2 cm of floodplain).

The figure compares the tables of the Cartesian 20 m cell in the middle of
the strip (in the channel row) with the terrain, integrated
directly (`cases/scripts/sgs_strip_tables.jl`; levels relative to the
floodplain at the cell centre).

In [ ]:
T = Dict(mesh => FC.sgs_strip_tables(FC.case_dir(ss), mesh) for mesh in ("cart20", "a5r18"))
c = T["cart20"]["cell"]; zf = c["z_floodplain"]
rel(v) = Float64.(v) .- zf
fig = Figure(size = (950, 720))
ax1 = Axis(fig[1, 1]; xlabel = "stored volume (m³)", ylabel = "water level − z_f (m)",
           title = @sprintf("Cell volume V(η), cell at x = %.0f m", c["x"]))
lines!(ax1, Float64.(c["V_analytic"]), rel(c["wse"]); color = FP.INK, linewidth = 3, label = "terrain (exact)")
lines!(ax1, Float64.(c["V_table"]), rel(c["wse"]); color = FP.SERIES[1], linewidth = 1.5, linestyle = :dash, label = "SGS table")
ax2 = Axis(fig[1, 2]; xlabel = "wetted area (m²)", ylabel = "water level − z_f (m)", title = "Cell wetted area")
lines!(ax2, Float64.(c["Awet_analytic"]), rel(c["wse"]); color = FP.INK, linewidth = 3, label = "terrain (exact)")
lines!(ax2, Float64.(c["Awet_table"]), rel(c["wse"]); color = FP.SERIES[1], linewidth = 1.5, linestyle = :dash, label = "SGS table")
for (k, (key, ttl)) in enumerate((("face_downstream", "downstream face (channel)"), ("face_lateral", "lateral face (to a floodplain row)")))
    e = c[key]
    ax = Axis(fig[2, k]; xlabel = "face flow area (m²)", ylabel = "water level − z_f (m)", title = "Flow area through the $ttl")
    lines!(ax, Float64.(e["A_analytic"]), rel(c["wse"]); color = FP.INK, linewidth = 3, label = "terrain (exact)")
    lines!(ax, Float64.(e["A_i"]), rel(c["wse"]); color = FP.SERIES[1], linewidth = 1.5, label = "table of this cell (A_i)")
    lines!(ax, Float64.(e["A_j"]), rel(c["wse"]); color = FP.SERIES[2], linewidth = 1.5, label = "table of the neighbour (A_j)")
    lines!(ax, Float64.(e["A_edge"]), rel(c["wse"]); color = FP.SERIES[3], linewidth = 2.5, linestyle = :dash, label = "used: ½(A_i + A_j)")
    k == 1 && axislegend(ax; position = :rb, framevisible = false)
end
for axx in contents(fig.layout)
    axx isa Axis || continue
    hlines!(axx, [c["z_max"] - zf]; color = FP.MUTED, linestyle = :dot)
    ylims!(axx, -1.05, 1.5)
end
text!(ax1, 5, c["z_max"] - zf; text = "cell's highest DEM sample (z_max)", align = (:left, :bottom), fontsize = 10, color = FP.INK2)
axislegend(ax1; position = :rb, framevisible = false)
fig

Over all cells and faces of the measuring reach, the table errors at levels
from 0.75 m below to 1 m above the floodplain (stored volume: the error
relative to the larger of V and 5 cm over the cell; faces: the median ratio of
the flow area the kernel uses to the face's true flow area, and the same for
the hydraulic radius):

In [ ]:
g(d, k) = Float64(get(d, k, NaN))
rows = []
for mesh in ("cart20", "a5r18"), δ in T[mesh]["levels"]
    t = T[mesh]; s = string(Float64(δ))
    push!(rows, (mesh == "cart20" ? "Cartesian 20 m" : "A5 res 18", f(δ, 2),
                 f(g(t["volume_rel_err"][s], "median"), 4), f(g(t["volume_rel_err"][s], "max_abs"), 3),
                 f(g(t["edge_area_ratio"]["channel"][s], "median"), 3), f(g(t["edge_area_ratio"]["floodplain"][s], "median"), 3),
                 f(g(t["edge_radius_ratio"]["channel"][s], "median"), 3), f(g(t["edge_radius_ratio"]["floodplain"][s], "median"), 3)))
end
mdtable(["mesh", "δ (m)", "V error (median)", "V error (max |·|)", "A ratio, channel faces", "A ratio, floodplain faces",
         "R ratio, channel faces", "R ratio, floodplain faces"], rows)

What this means for conveyance: the discharge one cross-section of the
Cartesian 20 m strip can carry at uniform flow on the bed slope, from the
tables as the kernel uses them (three faces, each at the upstream cell's
level), against the analytic conveyances of §3.

In [ ]:
u = T["cart20"]["uniform_flow"]
δs = range(-0.95, 3.0; length = 300)
Qof(mode; kw...) = [FC.sgs_strip_conveyance(p, δ, mode; kw...) * sqrt(p.S) / FC.SGS_STRIP_N for δ in δs]
fig = Figure(size = (800, 430))
ax = Axis(fig[1, 1]; xlabel = "uniform-flow discharge (m³/s)", ylabel = "level above the floodplain δ (m)",
          title = @sprintf("Discharge a cross-section carries at the bed slope, Cartesian 20 m (face at x = %.0f m)", u["x_face"]))
lines!(ax, Qof(:twod), δs; color = FP.INK, linewidth = 3, label = "2-D (fine-grid limit)")
lines!(ax, Qof(:lumped; rows = 20.0), δs; color = FP.SERIES[3], linewidth = 2, label = "lumped 20 m rows (exact SGS tables)")
lines!(ax, Qof(:cells; rows = 20.0), δs; color = FP.SERIES[2], linewidth = 1.5, linestyle = :dash, label = "cell-mean 20 m rows (standard)")
lines!(ax, Float64.(u["Q"]), Float64.(u["delta"]); color = FP.SERIES[1], linewidth = 2.5, label = "SGS tables as built")
vlines!(ax, [3.0, 20.0]; color = FP.MUTED, linestyle = :dot)
xlims!(ax, 0, 40); ylims!(ax, -1, 3)
axislegend(ax; position = :rb, framevisible = false)
@printf("SGS tables: at most %.2f m³/s at the bed slope (reached at δ = %.2f m); 20 m³/s needs a water-surface slope of %.4f (%.0f× the bed)\n",
        u["Q_max"], u["delta_at_Q_max"], p.S * (20 / u["Q_max"])^2, (20 / u["Q_max"])^2)
fig

## 5. Results

### Steady state in the measuring reach

Per run, at its last frame (8 h; 2.5 m runs 4 h), over x = 1000–2000 m: the
level above the floodplain in the channel cells and in the floodplain cells
(if wet), their spread over all wet cells, the storage per metre (water
volume, and the volume above the cell-mean bed), the storage relative to the
2-D answer, the storage change over the last hour as a fraction of the inflow
(0 when steady), and the arrival times. The analytic values to compare with
are in the table of §3.

In [ ]:
order = ["fine_std", "fine_swe", "cart20_std", "cart20_swe", "cart20_sgs", "cart20_sgspos", "a5r18_std", "a5r18_sgs"]
rows = []
for Q in QS, o in order
    j = o * "_" * qtag(Q); d = m[j]; q = qtag(Q)
    ref = startswith(o, "a5") ? "a5w_$q" : "2d_$q"     # A5: the 2-D answer for its own mean width
    dry(x) = isnan(x) ? "dry" : f(x)
    push!(rows, (label(j), f(d["delta_channel"]), dry(d["delta_floodplain"]), f(d["delta_spread"]),
                 f(d["store_per_m"], 2), f(d["store_above_mean_per_m"], 2), f(d["store_per_m"] / a["store_$ref"], 2),
                 f(d["dSdt_rel_end"], 3), isfinite(d["t_arrive_x1000"]) ? round(Int, d["t_arrive_x1000"]) : "never",
                 isfinite(d["t_arrive_x2000"]) ? round(Int, d["t_arrive_x2000"]) : "never", f(d["mass_balance_rel"], 12)))
    o == last(order) || continue
    push!(rows, ("analytic 2-D, $(Int(Q)) m³/s", f(a["delta_2d_$q"]), Q > 5 ? f(a["delta_2d_$q"]) : "dry", "0",
                 f(a["store_2d_$q"], 2), "", "1.0", "0", round(Int, a["t_arrive_2d_$(q)_x1000"]), round(Int, a["t_arrive_2d_$(q)_x2000"]), ""))
end
mdtable(["run", "δ channel (m)", "δ floodplain (m)", "spread (m)", "storage (m²/m)", "above mean bed (m²/m)",
         "÷ 2-D (A5: own width)", "dS/dt ÷ Q", "arrival 1000 m (s)", "2000 m (s)", "mass error"], rows)

### Long profiles

The level above the floodplain along the flume at the last frame, in the
channel cells (mean per 20 m of x): uniform flow is a horizontal line. The
outlet reach starts at x = 3800 m.

In [ ]:
"Mean δ of the wet channel cells per bin of x (m)."
function chan_profile(r; bin = 20.0)
    pr = FC.sgs_strip_profile(r, FA.nframes(r), p)
    edges = 0:bin:3800
    xs = Float64[]; δs = Float64[]
    for x0 in edges[1:end-1]
        sel = [i for i in eachindex(pr.x) if pr.chan[i] && pr.wet[i] && x0 <= pr.x[i] < x0 + bin]
        isempty(sel) && continue
        push!(xs, x0 + bin / 2); push!(δs, mean(pr.δ[sel]))
    end
    return xs, δs
end
style = Dict("fine_std" => (FP.INK, :solid, 3), "fine_swe" => (FP.INK2, :dash, 2),
             "cart20_std" => (FP.SERIES[2], :solid, 1.5), "cart20_swe" => (FP.SERIES[2], :dash, 1.5),
             "cart20_sgs" => (FP.SERIES[1], :solid, 2.5), "cart20_sgspos" => (FP.SERIES[1], :dot, 2.5),
             "a5r18_std" => (FP.SERIES[3], :solid, 1.5), "a5r18_sgs" => (FP.SERIES[4], :solid, 2.5))
fig = Figure(size = (950, 1050))
for (k, Q) in enumerate(QS)
    q = qtag(Q)
    ax = Axis(fig[k, 1]; xlabel = "x (m along the flume)", ylabel = "δ = η − z_f (m)",
              title = "Water level above the floodplain in the channel cells at the last frame, $(Int(Q)) m³/s")
    hspan!(ax, [-1.0], [0.0]; color = (FP.MUTED, 0.12))
    vspan!(ax, [FC.SGS_STRIP_REACH[1]], [FC.SGS_STRIP_REACH[2]]; color = (FP.SERIES[3], 0.07))
    for o in order
        c, ls, lw = style[o]; x, d = chan_profile(runs[o * "_" * q])
        lines!(ax, x, d; color = c, linestyle = ls, linewidth = lw, label = LABEL[o])
    end
    hlines!(ax, [a["delta_2d_$q"]]; color = FP.INK, linestyle = :dot, linewidth = 1)
    text!(ax, 3790, a["delta_2d_$q"]; text = "2-D normal level", align = (:right, :top), fontsize = 10, color = FP.INK2)
    ylims!(ax, Q < 5 ? (-0.45, 0.3) : (-0.2, 1.6)); xlims!(ax, 0, 3800)
end
Legend(fig[3, 1], contents(fig[1, 1])[1]; orientation = :horizontal, nbanks = 3, framevisible = false, labelsize = 11)
ax = Axis(fig[4, 1]; xlabel = "x (m along the flume)", ylabel = "δ (m)", yscale = Makie.pseudolog10,
          title = "The same at 20 m³/s on a log scale: the SGS runs (other runs as above)")
for o in ("cart20_sgs", "cart20_sgspos", "a5r18_sgs", "fine_std")
    c, ls, lw = style[o]; x, d = chan_profile(runs[o * "_q20"])
    lines!(ax, x, d; color = c, linestyle = ls, linewidth = lw, label = LABEL[o])
end
xlims!(ax, 0, 3800); axislegend(ax; position = :rt, framevisible = false, labelsize = 10)
fig

### Filling

Storage per metre in the measuring reach against time. A correct model fills
the reach as the kinematic front passes (§3 arrival times) and then holds the
normal-depth storage (dotted lines).

In [ ]:
function reach_storage(r)
    x, _ = xy(r); sel = [i for i in eachindex(x) if FC.SGS_STRIP_REACH[1] <= x[i] < FC.SGS_STRIP_REACH[2]]
    L = FC.SGS_STRIP_REACH[2] - FC.SGS_STRIP_REACH[1]
    return r.t ./ 3600, [sum(v -> isfinite(v) ? v : 0.0, FA.volume(r, k)[sel]) / L for k in 1:FA.nframes(r)]
end
fig = Figure(size = (950, 520))
for (k, Q) in enumerate(QS)
    q = qtag(Q)
    ax = Axis(fig[1, k]; xlabel = "time (h)", ylabel = "storage, x = 1000–2000 m (m²/m)",
              title = "Storage in the measuring reach, $(Int(Q)) m³/s", yscale = Q > 5 ? log10 : identity)
    for o in order
        c, ls, lw = style[o]; t, s = reach_storage(runs[o * "_" * q])
        Q > 5 && (s = max.(s, 1.0))
        lines!(ax, t, s; color = c, linestyle = ls, linewidth = lw, label = LABEL[o])
    end
    hlines!(ax, [a["store_2d_$q"]]; color = FP.INK, linestyle = :dot)
    k == 1 && ylims!(ax, 0, 20)
    k == 2 && ylims!(ax, 1, 300)
end
Legend(fig[2, 1:2], contents(fig[1, 1])[1]; orientation = :horizontal, nbanks = 2, framevisible = false, labelsize = 11)
fig

## 6. Why SGS holds water back

The strip separates three candidate causes — the tables, the flux
calculation, the storage accounting — because each can be checked on its own.

### 6.1 Storage accounting: not the cause

The SGS tables reproduce the stored volume of the terrain to 0.3% at every
level on the 20 m mesh, and to a median of 0.1% on A5 (§4). Overbank, SGS's
water volume and its volume above the cell-mean bed agree to within 1%
(the *measurement* check), as they must. SGS stores more because its water
level is higher, not because it counts water a standard grid cannot hold.

### 6.2 In bank: the legacy donor cap

SGS runs with `--limiter legacy` by default (`--scheme recommended` gives
positivity to the standard solver and legacy to SGS). Its donor cap allows an
edge to take at most V/(2N) of the donor cell's volume in one step
(`_face_flux_sgs!`, `surfacewater/kernels.jl`, the `!POS` branch; N = 4 faces
on a Cartesian grid, 5 on A5). A channel cell that must pass Q downstream
through one face can do so only when it holds V ≥ 2N·Q·Δt, however good its
tables. At 3 m³/s with Δt ≈ 4.3 s (the Courant limit for 1 m of water in a
20 m cell) that is ≈ 100 m³ — the brim-full channel cell, where the 2-D
normal depth needs 71 m³. The SGS run settles exactly there:

In [ ]:
# The legacy donor cap: an edge may move at most V_donor/(2N) per step (N = 4 faces on a Cartesian grid).
# If it binds, a channel cell passes Q only when it holds V ≥ 2N·Q·Δt.
for job in ("cart20_sgs_q3", "cart20_sgspos_q3")
    r = runs[job]; s = r.info["summary"]
    dt = r.t[end] / s["n_steps"]                      # mean time step (s)
    pr = FC.sgs_strip_profile(r, FA.nframes(r), p)
    sel = [i for i in eachindex(pr.x) if pr.chan[i] && FC.SGS_STRIP_REACH[1] <= pr.x[i] < FC.SGS_STRIP_REACH[2]]
    Vc = mean(pr.V[sel])
    @printf("%-34s mean Δt %.2f s; channel cell holds %.1f m³ (%.2f m²/m);  2N·Q·Δt = %.1f m³\n",
            label(job), dt, Vc, Vc / 20, 8 * 3.0 * dt)
end
@printf("for comparison: a 20 m channel cell at the 2-D normal depth holds %.1f m³; brim-full (δ = 0) %.1f m³\n",
        20 * FC.sgs_strip_storage(p, a["delta_2d_q3"]), 20 * FC.sgs_strip_storage(p, 0.0))

With `--limiter positivity` (one scale factor per donor cell, applied only
when its total outflow would exceed its volume), the same tables give the
lumped analytic level within 5 mm and its storage within 1% (1% above the
2-D answer). The cap is a
limiter artefact, not a property of the tables: it binds whenever a cell's
discharge times the time step is a sizeable fraction of its volume — deep,
fast channel flow through few faces — and it scales with Δt, so with cell
size. (`docs/HYDRAULICS.md` §15.7 found the SGS donor cap binding on half of
all wet edge-steps at Carlisle.)

### 6.3 Overbank: edge tables that stop at the cell's highest DEM sample

`build_sgs_tables!` (`mesh/A5Grid.jl`, step 2) tabulates each face's flow
area and wetted width at the *cell's own* 100 levels, from its z_min to its
z_max, and `edge_area_radius` (and `flow_area_from_wse`,
`wetted_perim_from_wse`) return the top value for any level above z_max. The
cell volume, by contrast, is extended above z_max (`wse_from_volume`). Three
consequences, all visible in §4:

1. **The flow area stops growing** once the water is above the cell's
   highest sample. On a 20 m cell of this strip that is 1 cm above the
   floodplain at the cell centre: a channel face is frozen at about bankfull
   (5 m² instead of 13 m² at δ = 0.4 m), a floodplain face at 0.2–0.4 m²
   instead of 8 m².
2. **Face points above the cell's z_max are never wet in that cell's
   table.** The downstream cell's table for a face at its upstream edge (the
   highest ground in that cell) sees only the channel part of the face, or
   nothing at all on the floodplain (A_j = 0, P_j = 0, hence R_j = 0).
3. **The kernel averages the two tables**, A = ½(A_i + A_j) and
   R = ½(R_i + R_j), so a zero from one side halves both.

At the bed slope the 20 m strip's tables can carry at most 4.9 m³/s (§4,
uniform flow). To pass 20 m³/s the water surface must steepen to about
17 times the bed slope, so the level rises towards the inflow without
limit and the flume keeps filling — which is what the SGS runs do. The same
holds on A5 (floodplain faces at 3% of their true area at δ = 0.4 m).

### 6.4 Is Carlisle in that regime?

If the `carlisle_peak` case has been run (`notebooks/03_carlisle.ipynb`), the
cell below counts, at its last frame (55.5 h, near the Eden peak), the wet
SGS cells whose water level is above their highest DEM sample — the cells
whose faces have frozen flow areas.

In [ ]:
cpk = FC.case("carlisle_peak")
pk = FC.job_output(cpk, only(filter(j -> j.name == "a5_sgs", cpk.jobs)))
if isfile(pk)
    r = FA.load_run(pk); k = FA.nframes(r)
    V = FA.volume(r, k); η = FA.wse(r, k); zmax = r.mesh.static["sgs_z_max"]; zmin = r.mesh.static["sgs_z_min"]
    wet = [isfinite(V[i]) && V[i] > 1e-3 * r.mesh.area[i] for i in eachindex(V)]
    up = wet .& (η .> zmax)
    @printf("carlisle_peak SGS at %.1f h: %d wet cells, %d (%.0f%%) with the water above their highest DEM sample,\n",
            r.t[k] / 3600, count(wet), count(up), 100 * count(up) / count(wet))
    @printf("  holding %.2f of the %.2f Mm³ stored; there the water is a median %.2f m above z_max, and the median relief\n",
            sum(V[up]) / 1e6, sum(V[wet]) / 1e6, median((η .- zmax)[up]))
    @printf("  z_max − z_min of the wet cells is %.2f m\n", median((zmax .- zmin)[wet]))
else
    println("carlisle_peak has not been run here ($pk)")
end

## 7. Conclusions

- **Storage is measured correctly.** SGS's tables hold the terrain's volume
  to 0.3% (median 0.05% on A5), and overbank the volume above the cell-mean
  bed equals the water volume — for SGS as for the 2.5 m reference. In bank,
  a cell-mean grid stores *more* than the true volume (2.2× here), the
  opposite of what SGS does at Carlisle. The extra SGS storage is real water,
  held back.
- **SGS conveyance is wrong in two ways, both independent of the storage
  tables:**
  1. *Overbank* — a face's flow area and wetted width stop growing once the
     water is above the cell's highest DEM sample, face points above a cell's
     z_max are never wet in that cell's table, and the kernel averages the
     two cells' values. On the 20 m strip the faces carry 39% (channel) and
     2.5% (floodplain) of their true flow area at 0.4 m above the floodplain,
     and the tables cannot pass more than 4.9 m³/s at the bed slope. At
     20 m³/s SGS piles the water up — 4.8 m above the floodplain in the
     measuring reach against 0.37 m, 3.9 times the storage, still filling at
     8 h, the front arriving 1.7–2.8 times later than in the reference.
     Code: `build_sgs_tables!` (`mesh/A5Grid.jl`, step 2: edge curves on the
     cell's own z_min–z_max levels), `edge_area_radius` /
     `flow_area_from_wse` / `wetted_perim_from_wse` (no extrapolation above
     the top level), and the ½(A_i + A_j), ½(R_i + R_j) average in
     `_face_flux_sgs!` (`surfacewater/kernels.jl`; also
     `surfacewater/reference_steps.jl`).
  2. *In bank* — the legacy donor cap, SGS's default limiter, lets a cell
     pass at most V/(2N) per step, so a channel cell conveys Q only once it
     holds 2N·Q·Δt: the channel runs brim-full (1.4 times the storage).
     `--limiter positivity` gives the right level with the same tables.
- **With correct tables** the SGS face formula (one hydraulic radius per
  face) reaches the analytic level in bank, and overbank the level of a
  cell-mean grid (+0.04 m, +8% storage on 20 m cells) — a limit of the
  formulation, much smaller than the defects.
- **Carlisle is in the overbank regime**: at the Eden peak almost all of the
  SGS water is in cells whose level is above their highest DEM sample
  (§6.4), which fits the 2.5× storage and the 3 m higher water surface.
- On **A5** the picture is the same (floodplain faces at 3% of their flow
  area at δ = 0.4 m), and SGS is worse than local-inertial on the same mesh
  even in bank (out of bank at 3 m³/s, still filling at 8 h).

These checks fail and are kept, as expected failures, until SGS is fixed;
the fix (edge curves over a range that covers both cells and extends above
it with the face's full width; possibly one curve per face shared by both
cells; positivity limiting for SGS) belongs behind a flag and should be
validated on this case first.

In [ ]:
# The headline numbers of §7 from the case's metrics (measuring reach, last frame)
for (job, ref) in (("cart20_sgs_q3", "2d_q3"), ("cart20_sgspos_q3", "2d_q3"), ("cart20_sgs_q20", "2d_q20"),
                   ("cart20_sgspos_q20", "2d_q20"), ("a5r18_sgs_q3", "2d_q3"), ("a5r18_sgs_q20", "a5w_q20"))
    d = m[job]
    @printf("%-42s δ %7.3f m (2-D %6.3f)   storage %7.2f m²/m = %5.2f× 2-D   dS/dt ÷ Q %5.2f\n", label(job),
            d["delta_channel"], a["delta_$ref"], d["store_per_m"], d["store_per_m"] / a["store_$ref"], d["dSdt_rel_end"])
end